In [9]:
import os
import random
import numpy as np
from PIL import Image
from tqdm import tqdm
from torchvision import transforms

In [10]:
SOURCE_DIR = "./dataset/Bijie-landslide-dataset"
DEST_DIR = "./dataset/bijie_binary_split"
OUTPUT_DIR = "./dataset/bijie_binary_dat"

TRAIN_RATIO = 0.7
SEED = 42

IMG_SIZE = 224

random.seed(SEED)

CLASSES = ["non-landslide", "landslide"]

CLASS_MAP = {
    "non-landslide": 0,
    "landslide": 1
}

In [11]:
import shutil

for split in ["train", "test"]:
    for cls in CLASSES:
        os.makedirs(os.path.join(DEST_DIR, split, cls), exist_ok=True)

def get_image_list(class_name):
    """
    Handles both:
    - Type 1: class/*.png
    - Type 2: class/image/*.png
    """
    base_dir = os.path.join(SOURCE_DIR, class_name)

    # Case 1: images directly inside folder
    files = [
        os.path.join(base_dir, f)
        for f in os.listdir(base_dir)
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".tif"))
    ]

    if len(files) > 0:
        return files

    # Case 2: look for /image subfolder
    image_dir = os.path.join(base_dir, "image")
    if os.path.exists(image_dir):
        return [
            os.path.join(image_dir, f)
            for f in os.listdir(image_dir)
            if f.lower().endswith((".jpg", ".jpeg", ".png", ".tif"))
        ]

    return []

def split_class(class_name):
    images = get_image_list(class_name)

    if len(images) == 0:
        print(f"⚠️ No images found for {class_name}")
        return

    random.shuffle(images)

    split_idx = int(len(images) * TRAIN_RATIO)
    train_imgs = images[:split_idx]
    test_imgs = images[split_idx:]

    for img_path in tqdm(train_imgs, desc=f"Train {class_name}"):
        fname = os.path.basename(img_path)

        shutil.copy(
            img_path,
            os.path.join(DEST_DIR, "train", class_name, fname)
        )

    for img_path in tqdm(test_imgs, desc=f"Test {class_name}"):
        fname = os.path.basename(img_path)

        shutil.copy(
            img_path,
            os.path.join(DEST_DIR, "test", class_name, fname)
        )

print("\n===== Splitting Dataset =====")
for cls in CLASSES:
    split_class(cls)


===== Splitting Dataset =====


Train non-landslide:   0%|          | 0/1402 [00:00<?, ?it/s]

Test landslide: 100%|██████████| 231/231 [00:00<00:00, 346.91it/s]


In [12]:
def count_images(base_dir):
    total = 0
    for cls in CLASSES:
        cls_dir = os.path.join(base_dir, cls)
        total += len(os.listdir(cls_dir))
    return total

TRAIN_DIR = os.path.join(DEST_DIR, "train")
TEST_DIR = os.path.join(DEST_DIR, "test")

N_TRAIN = count_images(TRAIN_DIR)
N_TEST = count_images(TEST_DIR)

print(f"\nTrain samples: {N_TRAIN}")
print(f"Test samples:  {N_TEST}")


Train samples: 1941
Test samples:  832


In [13]:
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [14]:
def write_memmap(base_dir, X_path, y_path, n_samples):
    X_mm = np.memmap(
        X_path,
        dtype="float16",
        mode="w+",
        shape=(n_samples, 3, IMG_SIZE, IMG_SIZE)
    )

    y_mm = np.memmap(
        y_path,
        dtype="int64",
        mode="w+",
        shape=(n_samples,)
    )

    idx = 0

    for cls, label in CLASS_MAP.items():
        cls_dir = os.path.join(base_dir, cls)

        for img_name in tqdm(os.listdir(cls_dir), desc=f"Writing {cls}"):
            img_path = os.path.join(cls_dir, img_name)

            try:
                img = Image.open(img_path).convert("RGB")
                img = transform(img).half().numpy()

                X_mm[idx] = img
                y_mm[idx] = label
                idx += 1

            except Exception as e:
                print(f"Skipping {img_path}: {e}")

    X_mm.flush()
    y_mm.flush()

    print(f"Saved {idx} samples → {X_path}")

In [15]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

write_memmap(
    TRAIN_DIR,
    f"{OUTPUT_DIR}/X_train.dat",
    f"{OUTPUT_DIR}/y_train.dat",
    N_TRAIN
)

write_memmap(
    TEST_DIR,
    f"{OUTPUT_DIR}/X_test.dat",
    f"{OUTPUT_DIR}/y_test.dat",
    N_TEST
)

Writing landslide: 100%|██████████| 539/539 [00:02<00:00, 218.96it/s]


Saved 1941 samples → ./dataset/bijie_binary_dat/X_train.dat


Writing landslide: 100%|██████████| 231/231 [00:00<00:00, 234.10it/s]


Saved 832 samples → ./dataset/bijie_binary_dat/X_test.dat


In [16]:
np.save(f"{OUTPUT_DIR}/meta_train.npy", {
    "shape": (N_TRAIN, 3, IMG_SIZE, IMG_SIZE),
    "dtype": "float16",
    "class_map": CLASS_MAP
})

np.save(f"{OUTPUT_DIR}/meta_test.npy", {
    "shape": (N_TEST, 3, IMG_SIZE, IMG_SIZE),
    "dtype": "float16",
    "class_map": CLASS_MAP
})

print("\n✅ DONE: Bijie binary dataset ready!")


✅ DONE: Bijie binary dataset ready!
